In [ ]:
import named_arrays as na
import astropy.units as u
import astropy.constants as const
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.colors as colors

import scipy.io

import ctis
import pathlib

from astropy.visualization import quantity_support

quantity_support()

In [ ]:
psf = 2.4 * u.arcsec / 2.35

# spectral_order = na.ScalarArray(np.array([7,7,3,1, -1]), axes='channel')
# angle = na.ScalarArray(np.array([0,120,240,0]),axes='channel') * u.deg

spectral_order = na.ScalarArray(np.array([1, 7, 8, -1, 3, 4, 1, 7, 8]), axes='channel')
angle = na.ScalarArray(np.array([0, 0, 0, 120, 120, 120, 240, 240, 240]), axes='channel') * u.deg

spatial_plate_scale = 0.59 * u.arcsec / u.pixel
spectral_plate_scale = 6.15 * u.AA / 1e3 / u.pixel / spectral_order

wv_0 = 16.78 * u.AA
thermal_width = 27 * u.km / u.s / const.c * wv_0
delta_lambda = 500 * u.km / u.s / const.c * wv_0


#note very small perturbation to grid by multiplying by .9999.  This works around a bug where conservative resampling
#is unstable to the grids points being exactly equal.
grid = na.SpectralPositionalVectorArray(
    wavelength=na.ScalarLinearSpace(wv_0 - delta_lambda, wv_0 + delta_lambda, axis='wavelength', num=101),
    position=na.Cartesian2dVectorArray(
        x=na.ScalarLinearSpace(-20, 20, axis='x', num=41) * u.pixel * spatial_plate_scale * .9999,
        y=na.ScalarLinearSpace(-20, 20, axis='y', num=41) * u.pixel * spatial_plate_scale * .9999,
    )
)

In [ ]:
scene = ctis.scenes.gaussians(grid, thermal_width)

In [ ]:
fig, ax = plt.subplots(2, 1)

intensity = scene.zeroth_moment('wavelength')
intensity_map_scaling = dict(vmin=1e-2, vmax=np.max(intensity.ndarray))

intensity_map = na.plt.pcolormesh(C=intensity, ax=ax[0], norm=colors.LogNorm(**intensity_map_scaling))
fig.colorbar(intensity_map.ndarray.item(), ax=ax[0])

velocity_map = na.plt.pcolormesh(C=scene.first_moment_velocity().value, alpha=intensity.ndarray / intensity.ndarray.max(), ax=ax[1],
                                 cmap='RdBu_r', vmin=-200, vmax=200)
fig.colorbar(velocity_map.ndarray.item(), ax=ax[1])